## Data Clean

In [14]:
%pip install git+https://github.com/jedisct1/draft-denis-ipcrypt.git#subdirectory=implementations/python

  Cloning https://github.com/jedisct1/draft-denis-ipcrypt.git to /tmp/pip-req-build-7km68v48
  Running command git clone --filter=blob:none --quiet https://github.com/jedisct1/draft-denis-ipcrypt.git /tmp/pip-req-build-7km68v48
  Resolved https://github.com/jedisct1/draft-denis-ipcrypt.git to commit 63b603f8222e1ac26147f69d5ddd487b8a6f0eaa
  Running command git submodule update --init --recursive -q
  Installing build dependencies ...done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import glob
from ipcrypt import deterministic # Deterministic Encryption
import ipaddress
from sklearn.preprocessing import LabelEncoder


In [16]:
path = "download_data/processed_data"
csv_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

df

,type,agentIP,inputPort,outputPort,srcMAC,dstMAC,ethType,inVlan,outVlan,srcIP,dstIP,ipProtocol,ipTos,ipTTL,srcPort,dstPort,tcpFlags,packetSize,payloadSize,skipCount
0,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,128.171.110.180,6,0x00,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0
1,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,128.171.110.180,6,0x00,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0
2,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,128.171.110.180,6,0x00,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0
3,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,128.171.110.180,6,0x00,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0
4,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,128.171.110.180,6,0x00,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32273286,FLOW,10.3.10.202,3,0,3cecefd4d083,ac74b1c0ad87,0x0800,301,0,128.171.177.51,10.3.31.121,6,0x00,53.0,443.0,51770.0,0x10,1522.0,1500.0,4096.0
32273287,FLOW,10.3.99.3,28,0,000ded4aacbf,3cecefc5a780,0x0800,110,0,128.171.165.6,128.171.110.132,6,0x48,57.0,22.0,33996.0,0x10,1522.0,1500.0,4096.0
32273288,FLOW,10.3.99.3,28,0,3cecefd4d083,f2b7fabb40fb,0x0800,301,0,172.67.158.111,10.3.31.93,17,0x00,53.0,443.0,61414.0,0x00,996.0,974.0,4096.0
32273289,FLOW,10.3.99.3,28,0,3cecefd4d083,f2b7fabb40fb,0x0800,301,0,172.67.158.111,10.3.31.93,17,0x00,53.0,443.0,61414.0,0x00,1250.0,1228.0,4096.0


In [17]:
# df = pd.read_csv("feb_to_may_flow_data.csv")

In [18]:
df = df.replace('-', np.nan)
df = df.dropna()

In [19]:
print(df['dstIP'].nunique()) # => 442
print(df['srcIP'].nunique()) # => 94819

442
94819


#### Before dropping 4th octet
- df['dstIP'].nunique() # => 289
- df['srcIP'].nunique() # => 9222

### IP Address to User Type Mapping

In [20]:
ip_matches = pd.read_csv("ip_user_map-v1.csv")
ip_matches_2 = pd.read_csv("HiloFlowIPMissingMap.csv")
ip_matches_2 = ip_matches_2.drop(columns=["hostname", "Who to ask"]).dropna()

ip_matches = ip_matches.rename(columns={
    "ipaddr": "missing_srcIP",
    "userclass": "UserClass"
})
ip_matches["missing_srcIP"] = ip_matches["missing_srcIP"].str.strip()
ip_matches_2["missing_srcIP"] = ip_matches_2["missing_srcIP"].str.strip()

# ip_matches.nunique()

In [21]:
ip_matches_concat = pd.concat([ip_matches,ip_matches_2], ignore_index=True).drop_duplicates()
ip_matches_concat

,missing_srcIP,UserClass
0,10.3.108.106,External
1,10.3.108.102,Admin
2,10.3.108.111,Executive
3,10.3.108.104,Admin
4,10.3.108.109,External
...,...,...
206,128.171.77.83,Phone
207,128.171.77.84,Phone
208,128.171.77.86,Phone
209,128.171.77.89,Phone


In [22]:
# match_prefixes = ip_matches["ipaddr"].str.split('.').str[0].tolist() # first octet

# match_prefixes = ip_matches['ipaddr']#.str.split('.').str[:2].str.join('.').tolist() # first 2 octets
# match_prefixes_2 = ip_matches_2['missing_srcIP']#.str.split('.').str[:2].str.join('.').tolist() # first 2 octets

# match_prefixes
# for i in range(len(match_prefixes) -  61 ):
#     print (match_prefixes[i])

In [23]:
df['srcIP'] = df['srcIP'].astype(str).str.strip()
df['dstIP'] = df['dstIP'].astype(str).str.strip()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32248645 entries, 0 to 32273290
Data columns (total 20 columns):
 #   Column       Dtype  
---  ------       -----  
 0   type         object 
 1   agentIP      object 
 2   inputPort    int64  
 3   outputPort   int64  
 4   srcMAC       object 
 5   dstMAC       object 
 6   ethType      object 
 7   inVlan       int64  
 8   outVlan      int64  
 9   srcIP        object 
 10  dstIP        object 
 11  ipProtocol   int64  
 12  ipTos        object 
 13  ipTTL        float64
 14  srcPort      float64
 15  dstPort      float64
 16  tcpFlags     object 
 17  packetSize   float64
 18  payloadSize  float64
 19  skipCount    float64
dtypes: float64(6), int64(5), object(9)
memory usage: 5.0+ GB


### IP Crypt Deterministic IP Obfuscation (New Approach)

In [24]:
key = bytes.fromhex("0123456789abcdeffedcba9876543210")


df['srcIP_obfuscated'] = df['srcIP'].apply(lambda x: deterministic.encrypt(x, key))
df['dstIP_obfuscated'] = df['dstIP'].apply(lambda x: deterministic.encrypt(x, key))

In [12]:
df['srcIP_obfuscated'] = df['srcIP_obfuscated'].apply(lambda x: int(ipaddress.IPv6Address(x)))

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32248645 entries, 0 to 32273290
Data columns (total 22 columns):
 #   Column            Dtype  
---  ------            -----  
 0   type              object 
 1   agentIP           object 
 2   inputPort         int64  
 3   outputPort        int64  
 4   srcMAC            object 
 5   dstMAC            object 
 6   ethType           object 
 7   inVlan            int64  
 8   outVlan           int64  
 9   srcIP             object 
 10  dstIP             object 
 11  ipProtocol        int64  
 12  ipTos             object 
 13  ipTTL             float64
 14  srcPort           float64
 15  dstPort           float64
 16  tcpFlags          object 
 17  packetSize        float64
 18  payloadSize       float64
 19  skipCount         float64
 20  srcIP_obfuscated  object 
 21  dstIP_obfuscated  object 
dtypes: float64(6), int64(5), object(11)
memory usage: 5.5+ GB


### IP Address Masking (Outdated)

In [26]:
# def mask_ip_addr(src_ip: str, dst_ip: str, match_prefixes: list[str]):

#     # Helper function to determine whether an ip matches one in the given list
#     def matches(ip):
#         for prefix in match_prefixes:
#             if ip.startswith(prefix):
#                 return True
#         return False

#     # Check for src ip case => mask src ip
#     if matches(src_ip):
#         return "0.0.0.0", dst_ip

#     # Check for dst ip case => mask dst ip
#     if matches(dst_ip):
#         return src_ip, "0.0.0.0"

#     # No matches => return both IPs as is
#     return src_ip, dst_ip



### AI test function with test cases (Outdated)

In [27]:
# # Test prefixes
# prefixes = ['192.168', '10.0']

# # Test cases: (src_ip, dst_ip)
# test_cases = [
#     ('192.168.1.10', '8.8.8.8'),   # src matches → mask src
#     ('8.8.8.8', '10.0.0.5'),       # dst matches → mask dst
#     ('172.16.0.3', '8.8.4.4'),     # no match → no masking
#     ('10.0.1.2', '10.0.2.3'),      # src matches → mask src
#     ('172.16.0.3', '10.0.1.5')     # dst matches → mask dst
# ]

# # Run the function on all test cases
# for src, dst in test_cases:
#     masked_src, masked_dst = mask_ip_addr(src, dst, prefixes)
#     print(f"Original: {src} → {dst} | Masked: {masked_src} → {masked_dst}")

### Creating new columns with IPs masked (Outdated)

In [28]:
# df[['masked_srcIP', 'masked_dstIP']] = df.apply(
#     lambda row: pd.Series(mask_ip_addr(row['srcIP'], row['dstIP'], match_prefixes)),
#     axis=1
# )

# df.head(20)

### Mapping userclass to srcIPs

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32248645 entries, 0 to 32273290
Data columns (total 22 columns):
 #   Column            Dtype  
---  ------            -----  
 0   type              object 
 1   agentIP           object 
 2   inputPort         int64  
 3   outputPort        int64  
 4   srcMAC            object 
 5   dstMAC            object 
 6   ethType           object 
 7   inVlan            int64  
 8   outVlan           int64  
 9   srcIP             object 
 10  dstIP             object 
 11  ipProtocol        int64  
 12  ipTos             object 
 13  ipTTL             float64
 14  srcPort           float64
 15  dstPort           float64
 16  tcpFlags          object 
 17  packetSize        float64
 18  payloadSize       float64
 19  skipCount         float64
 20  srcIP_obfuscated  object 
 21  dstIP_obfuscated  object 
dtypes: float64(6), int64(5), object(11)
memory usage: 5.5+ GB


In [30]:
ip_matches_concat.info()

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 0 to 210
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   missing_srcIP  210 non-null    object
 1   UserClass      210 non-null    object
dtypes: object(2)
memory usage: 4.9+ KB


In [31]:
ip_to_userclass = dict(zip(ip_matches_concat["missing_srcIP"], ip_matches_concat["UserClass"]))

# print(ip_to_userclass)
# map userclass to srcIP
df["userclass"] = df["srcIP"].map(ip_to_userclass)
df.head()

,type,agentIP,inputPort,outputPort,srcMAC,dstMAC,ethType,inVlan,outVlan,srcIP,...,ipTTL,srcPort,dstPort,tcpFlags,packetSize,payloadSize,skipCount,srcIP_obfuscated,dstIP_obfuscated,userclass
0,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,...,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,NaN
1,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,...,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,NaN
2,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,...,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,NaN
3,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,...,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,NaN
4,FLOW,10.3.99.3,28,0,000ded4aacbf,d4a2cdbe9599,0x0800,110,0,128.171.165.71,...,120.0,16000.0,57888.0,0x10,1522.0,1500.0,4096.0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,NaN


In [32]:
df_userclass = df[df["userclass"].notna()]
# df_userclass.info()
df_userclass["userclass"].value_counts()
df_userclass

# df_userclass.to_csv("userclasses_mapped_v2.csv", index=False)

,type,agentIP,inputPort,outputPort,srcMAC,dstMAC,ethType,inVlan,outVlan,srcIP,...,ipTTL,srcPort,dstPort,tcpFlags,packetSize,payloadSize,skipCount,srcIP_obfuscated,dstIP_obfuscated,userclass
96,FLOW,10.3.10.202,3,0,3cecefd4d083,3ceceffad800,0x0800,10,0,10.3.253.2,...,127.0,49560.0,22.0,0x18,126.0,104.0,4096.0,339f:82eb:461e:4d44:5d54:f5ec:7124:bbb6,feb7:b412:7050:38c5:f043:a9e4:a24f:72a0,RemoteControl
274,FLOW,10.3.99.3,28,0,0cc47aaa3b96,80ee73db5f74,0x0800,110,0,128.171.110.221,...,64.0,2049.0,814.0,0x10,1502.0,1480.0,4096.0,eefc:6e13:9b74:4671:1c3f:37e6:d1c4:aafb,9658:374d:b657:b424:aa1e:2e2e:4dbf:c5b6,Repository
311,FLOW,10.3.10.202,3,0,3cecefd4d083,0014fd1984e4,0x0800,108,0,128.171.168.137,...,253.0,123.0,123.0,0x00,98.0,76.0,4096.0,8775:f0ab:3e6a:8ae5:ab8:8ae4:82e7:ec78,ab46:eacf:bc98:6f3c:e088:5ccf:5bf3:8f89,Infrastructure
326,FLOW,10.3.99.3,28,0,0cc47aaa3b96,80ee73db5f74,0x0800,110,0,128.171.110.221,...,64.0,2049.0,814.0,0x10,1502.0,1480.0,4096.0,eefc:6e13:9b74:4671:1c3f:37e6:d1c4:aafb,9658:374d:b657:b424:aa1e:2e2e:4dbf:c5b6,Repository
333,FLOW,10.3.10.202,3,0,3cecefd4d083,ac74b1c0ad87,0x0800,301,0,128.171.169.199,...,61.0,80.0,49241.0,0x10,1522.0,1500.0,4096.0,5ad:561c:416a:21b:763:e1ec:29b3:dcba,29df:d73b:c569:eeec:8656:efda:ca45:5aae,Infrastructure
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32273272,FLOW,10.3.99.3,28,0,3cecefd4d083,ac74b1c0ad87,0x0800,301,0,128.171.226.68,...,53.0,443.0,51732.0,0x10,1522.0,1500.0,4096.0,1576:2c4:4988:6f7f:6a32:753a:bcf8:e711,29df:d73b:c569:eeec:8656:efda:ca45:5aae,Infrastructure
32273273,FLOW,10.3.10.202,3,0,3cecefd4d083,ac74b1c0ad87,0x0800,301,0,128.171.177.51,...,53.0,443.0,51740.0,0x10,64.0,42.0,4096.0,367b:875a:5ac0:37d2:215a:90bf:722f:81b5,29df:d73b:c569:eeec:8656:efda:ca45:5aae,Infrastructure
32273279,FLOW,10.3.99.3,28,0,000ded4aacbf,244bfed1d0d8,0x0800,110,0,128.171.2.5,...,242.0,53.0,37341.0,0x00,283.0,261.0,4096.0,aa46:6fca:13ef:4b7e:7140:ed84:3f3:ddc1,b822:d2c7:71e2:c29e:c69b:4bd:745f:22b1,Infrastructure
32273285,FLOW,10.3.99.3,28,0,3cecefd4d083,ac74b1c0ad87,0x0800,301,0,128.171.177.51,...,53.0,443.0,51770.0,0x10,1522.0,1500.0,4096.0,367b:875a:5ac0:37d2:215a:90bf:722f:81b5,29df:d73b:c569:eeec:8656:efda:ca45:5aae,Infrastructure


In [33]:
# df.head(60)
combined_ips = set(df["srcIP"]).union(set(df["dstIP"]))
matching_ips = combined_ips.intersection(set(ip_matches_concat["missing_srcIP"]))

missing_src_ips = df.loc[~df["srcIP"].isin(ip_matches_concat["missing_srcIP"]), "srcIP"].unique()
num_unique_src_ips = df["srcIP"].nunique()
num_matching_ips = len(matching_ips)
print("df unique ips:", num_unique_src_ips)
print("matching ips:", num_matching_ips)
print("missing src ips", missing_src_ips)

df unique ips: 94819
matching ips: 185
missing src ips ['128.171.165.71' '23.61.28.175' '128.171.165.7' ... '185.206.24.45'
 '34.98.113.125' '150.5.130.223']


### Curt wanted IPs that didn't map to user type

In [34]:
# Convert missing IPs to csv
# pd.DataFrame({"missing_srcIP": list(missing_src_ips)}).to_csv("missing_ips.csv", index=False)

### Converts IPs from strings to floats

In [35]:
import ipaddress
# Bitwise operation shift to turn IPs into float dtype
def ip_to_float(ip):
    try:
        return float(int(ipaddress.ip_address(ip)))
    except:
        return None
        
# df['masked_srcIP'] = df['masked_srcIP'].apply(ip_to_float)
# df['masked_dstIP'] = df['masked_dstIP'].apply(ip_to_float)

In [36]:
# Convert tcpFlags from hexadecimal => int64 values
# df['tcpFlags'] = df['tcpFlags'].apply(lambda x : int(x, 16)).astype('uint8')

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32248645 entries, 0 to 32273290
Data columns (total 23 columns):
 #   Column            Dtype  
---  ------            -----  
 0   type              object 
 1   agentIP           object 
 2   inputPort         int64  
 3   outputPort        int64  
 4   srcMAC            object 
 5   dstMAC            object 
 6   ethType           object 
 7   inVlan            int64  
 8   outVlan           int64  
 9   srcIP             object 
 10  dstIP             object 
 11  ipProtocol        int64  
 12  ipTos             object 
 13  ipTTL             float64
 14  srcPort           float64
 15  dstPort           float64
 16  tcpFlags          object 
 17  packetSize        float64
 18  payloadSize       float64
 19  skipCount         float64
 20  srcIP_obfuscated  object 
 21  dstIP_obfuscated  object 
 22  userclass         object 
dtypes: float64(6), int64(5), object(12)
memory usage: 5.8+ GB


In [38]:
# df['dstIP'].nunique() => 15
# df['srcIP'].nunique() => 4271
# df['srcMAC'].nunique() # => 64 unique devices potentially

In [39]:
# Total number of unique interfaces
unique_macs = pd.concat([df['srcMAC'], df['dstMAC']]).nunique()
unique_macs

721

In [40]:
# Keeping necessary columns for analysis & model training
keep_cols = ['srcIP_obfuscated', 'dstIP_obfuscated', 'srcPort', 'dstPort', 'ipProtocol', 'tcpFlags', 'packetSize', 'payloadSize', 'ipTTL']
# keep_cols = ['srcPort', 'dstPort', 'ipProtocol', 'tcpFlags', 'packetSize', 'payloadSize', 'ipTTL']

new_df = df[keep_cols]
new_df

,srcIP_obfuscated,dstIP_obfuscated,srcPort,dstPort,ipProtocol,tcpFlags,packetSize,payloadSize,ipTTL
0,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,16000.0,57888.0,6,0x10,1522.0,1500.0,120.0
1,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,16000.0,57888.0,6,0x10,1522.0,1500.0,120.0
2,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,16000.0,57888.0,6,0x10,1522.0,1500.0,120.0
3,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,16000.0,57888.0,6,0x10,1522.0,1500.0,120.0
4,f5a3:e0b7:cb5c:f698:dc21:29c2:4450:ebd4,989b:42df:cde0:d99:66b1:4ba2:5586:7bcc,16000.0,57888.0,6,0x10,1522.0,1500.0,120.0
...,...,...,...,...,...,...,...,...,...
32273286,367b:875a:5ac0:37d2:215a:90bf:722f:81b5,29df:d73b:c569:eeec:8656:efda:ca45:5aae,443.0,51770.0,6,0x10,1522.0,1500.0,53.0
32273287,7e2c:663e:8f1b:3045:78d3:10a3:78f7:5ee2,c97c:b245:f78d:34ca:b05:8be8:be9d:5500,22.0,33996.0,6,0x10,1522.0,1500.0,57.0
32273288,55c2:eda4:82ac:d9e8:4903:b147:e6fd:1a15,a37:a2a0:f08f:5347:b111:b57b:66c8:73b4,443.0,61414.0,17,0x00,996.0,974.0,53.0
32273289,55c2:eda4:82ac:d9e8:4903:b147:e6fd:1a15,a37:a2a0:f08f:5347:b111:b57b:66c8:73b4,443.0,61414.0,17,0x00,1250.0,1228.0,53.0


In [41]:
# Convert tcpFlags from hexadecimal => int64 values
# new_df['tcpFlags'] = new_df['tcpFlags'].apply(lambda x : int(x, 16)).astype('uint8')

In [42]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32248645 entries, 0 to 32273290
Data columns (total 9 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcIP_obfuscated  object 
 1   dstIP_obfuscated  object 
 2   srcPort           float64
 3   dstPort           float64
 4   ipProtocol        int64  
 5   tcpFlags          object 
 6   packetSize        float64
 7   payloadSize       float64
 8   ipTTL             float64
dtypes: float64(5), int64(1), object(3)
memory usage: 2.4+ GB


In [43]:
# Convert Object dtypes to labels
categorical_cols = new_df.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    new_df[col] = le.fit_transform(new_df[col].astype(str))
    label_encoders[col] = le

/tmp/ipykernel_168/1178124892.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[col] = le.fit_transform(new_df[col].astype(str))
/tmp/ipykernel_168/1178124892.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df[col] = le.fit_transform(new_df[col].astype(str))
/tmp/ipykernel_168/1178124892.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://panda

In [ ]:
new_df = new_df.dropna()

# new_df.info()
new_df.shape

In [ ]:
new_df.info()

## To do:

### Data aggregation to improve clustering
    - Take full list and try to map user roles
    - Hypothesis => sample 5-10 members of each cluster to find patterns that emerge.
    - Hypothesis: Clusters 0-3 will be relatively similar, with maybe 1 or 2 features that cause the difference in clusters, but cluster 4 will have very distinct data from the rest of the groups.

## K-Means Model 1 PCA

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

## Maybe the IPs 

In [ ]:
# Fits data to scale so no column is overpowering => helps perform unbiased clustering
scaler = StandardScaler()
fit_data_to_scale = scaler.fit_transform(new_df)

In [ ]:
# Create a KMeans model that will group data into X clusters
# random_state=42 makes sure we get the same result every time we run it
kmeans_pca = KMeans(n_clusters=8,random_state=42)

# Fit the model to the scaled data, computer studies the data and looks for patterns
# Then it assigns each data point to one of the 4 clusters
cluster_labels = kmeans_pca.fit_predict(fit_data_to_scale)

# Add the cluster results as a new column called 'cluster' into the df => we can see which group each row of data belongs to
new_df.loc[:, 'cluster'] = cluster_labels

In [ ]:
# Create a PCA model that reduces data down to 3 main components, "n_components=3" means we want to keep 3 new simplified features
# Hence "PCA Reduced"
pca = PCA(n_components=3)

# Fit the PCA model to the scaled data, it studies the data to find the most important patterns
# Then transforms the original data into 3 new columns that summarize most important information
X_pca = pca.fit_transform(fit_data_to_scale)

In [ ]:
# Plot data points
plt.figure(figsize=(10,6))
scatter = plt.scatter(X_pca[:,0],X_pca[:,1], c=cluster_labels, cmap='viridis', alpha=0.7)
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('K-means Clustering on Hilo Data')

clusters = np.unique(cluster_labels)
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=scatter.cmap(scatter.norm(cluster)),
               markersize=8,
               label=f'Cluster {cluster}')
    for cluster in clusters
]

plt.legend(handles=handles, title="Clusters")

plt.colorbar(scatter)
plt.show()

#### Zooming in on axes, to look closely at the non-outlier clusters

In [ ]:
# Plot data points
plt.figure(figsize=(10,6))
scatter = plt.scatter(X_pca[:,0],X_pca[:,1], c=cluster_labels, cmap='viridis', alpha=0.7)
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('K-means Clustering on Hilo Data')

clusters = np.unique(cluster_labels)
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=scatter.cmap(scatter.norm(cluster)),
               markersize=8,
               label=f'Cluster {cluster}')
    for cluster in clusters
]

plt.legend(handles=handles, title="Clusters")

plt.colorbar(scatter)

plt.xlim(-10,10)
plt.ylim(-10,10)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score
# Calculate silhouette score => measure how well each data point fits within its assigned cluster
# compared to other clusters

# increase silhouette sample size 10000 => 20000 => 50000 => 100000
silh_score = silhouette_score(X_pca, cluster_labels, sample_size=500000, random_state=42)

# Higher silh score => clusters well separated
silh_score

#### Sampling 5 data points from each cluster

In [ ]:
count_unique = new_df['cluster'].nunique() # sanity check

sampling_data = (new_df.groupby('cluster').apply(lambda x: x.sample(5, random_state=42), include_groups=False))
sampling_data

In [ ]:
# Week 11 To Do:
# Remap IP address using library => call remaining IPs (non zeros eg: 0.0.0.0) using function 
# High dst is client and source is server => eg: laptop is dst, facebook is src => port number is always high when we are client getting data from internet
# Think abt possible combinations the model can pick up when we train model on a df that maps with user ips and user type
# Remove cluster 4 from data => replot to expand plot and see if there's distinct clustering groups => limit x and y delims is another approach
# Work on t-sne pulling samples from clusters as well
# Work on slides relating to what we've done so far

## K-Means Model 2 tSNE

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
# Randomly set 10000 samples from df because tSNE is slow
tsne_sample_data = new_df.sample(n=100000, random_state=42)
tsne_sample_data = tsne_sample_data.drop(columns='cluster')

# Scale  data so no column is overpowering => ensures all featuers contribute to model equally
scaler = StandardScaler()

# Fit the scaler to the sampled data and transform the data
# fit calculates the mean and standard deviation for each column
# transform applies the scaling formula to standardize the data
scaled_data = scaler.fit_transform(tsne_sample_data)

In [ ]:
# Create tSNE object => tSNE reduces high dimensionality data into low dimensions
# Here I reduce data to 2 components
tsne = TSNE(n_components=2,
            random_state=42,
            perplexity=30, # Controls how t-SNE balances attention between local and global structure
            learning_rate=200) # Controls how quickly the algorithm adjusts positions during optimization


# Fits tSNE to scaled data and transforms it
# fit learns how to map high dimensional data into 2D (n=2)
# transform applies the mapping
X_tsne = tsne.fit_transform(scaled_data)

In [ ]:
# Create KMeans model
kmeans_tsne = KMeans(n_clusters=3, random_state=42)

# Fit KMeans model to tSNE data and assign cluster labels
cluster_labels_tsne = kmeans_tsne.fit_predict(X_tsne)

tsne_sample_data.loc[:, 'cluster'] = cluster_labels_tsne

In [ ]:
sample_df = tsne_sample_data.copy()

# Plot data points
plt.figure(figsize=(8,6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels_tsne, cmap='tab10', s=30)
plt.title('t-SNE Visualization (Sample of 100000 rows, 3 clusters)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.show()

#### Sampling 5 data points from each clsuter

In [ ]:
count_unique_2 = sample_df['cluster'].nunique()
# sample_df
count_unique_2

In [ ]:
sampling_data_tsne = (sample_df.groupby('cluster').apply(lambda x: x.sample(5), include_groups=False))
sampling_data_tsne